# Phase 2 — unchanged Qwen3-4B on DEV (PREREGISTRATION §5 with Amendment 1)
Run this **once, before any training run**. 40 DEV families × E/R/C = 120 generations with the unchanged model.
Frozen protocol (PREREGISTRATION §4): prompt `test1-english-final-v2`, greedy non-thinking decoding, NF4 + FP16, SDPA math, output cap 1,024 tokens, scorer v1.0.0, engine seed 11. **Do not change any setting.**
The notebook embeds `phase2/IMPLEMENTATION_MANIFEST.md` (SHA-256 `6a6370cf9bb8dec6d31a020031bb3ec389c5d5ae4b89fcece465b2464918ab19`) and refuses to run if any embedded code or data differs from it, or if the pinned tokenizer/config files differ (Amendment 1 item 14).
At the end it prints **high unchanged-DEV accuracy (ceiling-risk proxy)** with k/40 and its Wilson 95% interval; the flag fires at C ≥ 38/40 (Amendment 1 item 7). It is a proxy only: Δ_C stays first in the report whatever it shows. DEV is a sanity set: it never selects anything. No FINAL_TEST data is in this notebook.
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (about 15–20 minutes). Drive is on by default so a disconnect resumes (re-run all; finished prompts are kept, never regenerated). A results ZIP with checksums downloads at the end; send it unchanged.


In [ ]:
import os, sys, json, base64, io, zipfile, hashlib, subprocess, time
from pathlib import Path
USE_DRIVE = True  # resume after a disconnect
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'  # download bars would become thousands of streamed lines
PACKAGE = Path('/content/hinglish_phase2') if Path('/content').exists() else Path.cwd() / 'hinglish_phase2'
PACKAGE.mkdir(parents=True, exist_ok=True)
blob = base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest() == 'd37ab2e1d6a6bc2b3d9641ea7c5d0276b11639f13f9490bf8cd5e053d6cd224b'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(PACKAGE)
INPUTS = PACKAGE / 'inputs'
# Implementation manifest (PREREGISTRATION Amendment 1 item 14): refuse to run if the embedded code differs from it.
MANIFEST_TEXT = base64.b64decode('IyBQaGFzZS0yIGltcGxlbWVudGF0aW9uIG1hbmlmZXN0IChQUkVSRUdJU1RSQVRJT04gQW1lbmRtZW50IDEgaXRlbSAxNCkKCkdlbmVyYXRlZCBieSBgcHl0aG9uMyAtQiBwaGFzZTIvYnVpbGRfaW1wbGVtZW50YXRpb25fbWFuaWZlc3QucHlgOyB2ZXJpZnkgd2l0aCBgLS1jaGVja2AuIERvIG5vdCBlZGl0IGJ5IGhhbmQuCgotIENvbW1pdCB0aGlzIGZpbGUgKipiZWZvcmUgdGhlIGZpcnN0IFBoYXNlLTIgREVWIGdlbmVyYXRpb24qKi4gVGhlIERFVi1iYXNlbGluZSBhbmQgVHJhaW4gbm90ZWJvb2tzIGVtYmVkIGl0IHdpdGggaXRzIFNIQS0yNTYgYW5kIHJlZnVzZSB0byBydW4gaWYgYW55IGVtYmVkZGVkIGNvZGUgb3IgZGF0YSBkaWZmZXJzIGZyb20gaXQuCi0gSXQgaXMgcmVjaGVja2VkIGF0IGFkYXB0ZXIgZnJlZXplOiBldmVyeSByZXRhaW5lZCBydW4gcmVjb3JkcyB0aGlzIGZpbGUncyBTSEEtMjU2LCBhbmQgYHBoYXNlMi9BREFQVEVSU19GUk9aRU4ubWRgIGFuZCB0aGUgRmluYWxUZXN0IGdhdGUgcmVxdWlyZSBpdC4KLSBBbnkgY2hhbmdlIHRvIGEgaGFzaGVkIGZpbGUgbmVlZHMgYSByZWdlbmVyYXRlZCBtYW5pZmVzdCBhbmQgcmVidWlsdCBub3RlYm9va3MgYmVmb3JlIHVzZS4KCmBgYGpzb24KewogInNjaGVtYSI6ICJwaGFzZTItaW1wbGVtZW50YXRpb24tbWFuaWZlc3QtMSIsCiAicHJlcmVnaXN0cmF0aW9uIjogewogICJwYXRoIjogInBoYXNlMi9QUkVSRUdJU1RSQVRJT04ubWQiLAogICJzaGEyNTYiOiAiMzdlZjY4YmZjYjdjZmEwYmI1MzBiZmJlY2JmMjA4MmQxOGU5MTZjMDIxNzM3MDA4NWQ0MTgwZThlYTc1OWFkYSIsCiAgImluY2x1ZGVzIjogIkFtZW5kbWVudCAxIChEMDQ1KSIKIH0sCiAiZ3JvdXBzIjogewogICJ0cmFpbmVyIjogWwogICAidDRfc21va2UvdHJhaW5fcGhhc2UyLnB5IiwKICAgInQ0X3Ntb2tlL3RyYWluX3Ntb2tlLnB5IgogIF0sCiAgImxhYmVsX2J1aWxkZXIiOiBbCiAgICJ0NF9zbW9rZS9sYWJlbHMucHkiCiAgXSwKICAicHJvbXB0IjogWwogICAidDRfc21va2UvcHJvdG9jb2wucHkiCiAgXSwKICAiZXZhbHVhdGlvbl9kcml2ZXIiOiBbCiAgICJ0NF9zbW9rZS9waGFzZTJfZXZhbC5weSIKICBdLAogICJlbmdpbmUiOiBbCiAgICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9fX2luaXRfXy5weSIsCiAgICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9kYXRhLnB5IiwKICAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L2VuZ2luZS5weSIsCiAgICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9oZl9iYWNrZW5kLnB5IiwKICAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L3JlcG9ydC5weSIsCiAgICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9zY29yZXJfdjEucHkiCiAgXSwKICAic2NvcmVyX3YxIjogWwogICAidGVzdDFfcGlsb3QvaGluZ2xpc2hfcGlsb3Qvc2NvcmVyX3YxLnB5IgogIF0sCiAgIm51bWVyaWNfbGl0ZXJhbF92MSI6IFsKICAgInNlY29uZGFyeV9hbmFseXNpcy9udW1lcmljX2VuZHBvaW50LnB5IgogIF0sCiAgIm51bWVyaWNfbGl0ZXJhbF92MiI6IFsKICAgInNlY29uZGFyeV9hbmFseXNpcy9udW1lcmljX2xpdGVyYWxfdjIucHkiCiAgXSwKICAic2Vjb25kYXJ5X3JlcG9ydCI6IFsKICAgInNlY29uZGFyeV9hbmFseXNpcy9fX2luaXRfXy5weSIsCiAgICJzZWNvbmRhcnlfYW5hbHlzaXMvY21pLnB5IiwKICAgInNlY29uZGFyeV9hbmFseXNpcy9yZXBvcnRfc2Vjb25kYXJ5LnB5IgogIF0sCiAgImFuYWx5c2lzIjogWwogICAicGhhc2UyX2FuYWx5c2lzL19faW5pdF9fLnB5IiwKICAgInBoYXNlMl9hbmFseXNpcy9hbmFseXplLnB5IiwKICAgInBoYXNlMl9hbmFseXNpcy9jYWxpYnJhdGUucHkiLAogICAicGFwZXJfYW5hbHlzaXMvc2FmZXppcC5weSIKICBdLAogICJub3RlYm9va19idWlsZGVycyI6IFsKICAgImNvbGFiX25vdGVib29rcy9idWlsZF9waGFzZTJfbm90ZWJvb2tzLnB5IiwKICAgInBoYXNlMi9idWlsZF9pbXBsZW1lbnRhdGlvbl9tYW5pZmVzdC5weSIKICBdLAogICJleHBvc3VyZV9vcmRlciI6IFsKICAgInBoYXNlMi9FWFBPU1VSRV9PUkRFUi5qc29uIgogIF0sCiAgImZyb3plbl9kYXRhIjogWwogICAicGhhc2UyL3RyYWluX0EudjIuanNvbiIsCiAgICJwaGFzZTIvdHJhaW5fQi52Mi5qc29uIiwKICAgInBoYXNlMi9kZXYudjIuanNvbiIKICBdCiB9LAogImZpbGVzIjogewogICJjb2xhYl9ub3RlYm9va3MvYnVpbGRfcGhhc2UyX25vdGVib29rcy5weSI6ICIyNDRmNGUyODBmNTZmNThjOWU1NDIzZWNhNGU3MWE3NmI4ZjFiNThkNmUxMjNiOTYzYzE0MzY1YzliMDliNTVmIiwKICAicGFwZXJfYW5hbHlzaXMvc2FmZXppcC5weSI6ICJmMDk3YTliODFhNjhmMzU2NGMyN2FlNjBkMzhhNTNjODM0Y2VlNWMzOGI0YTIwYjczOWRmYWUwYzNkZDY1NTNkIiwKICAicGhhc2UyL0VYUE9TVVJFX09SREVSLmpzb24iOiAiYWRmNDNlNTBjODBjNTlmZmE4MzEwNzE1NTg3ODU0NjliMWVkYTg5ODgwYzE4ZWVmYjRmODlhOWFkZWI1YWZiNiIsCiAgInBoYXNlMi9idWlsZF9pbXBsZW1lbnRhdGlvbl9tYW5pZmVzdC5weSI6ICIxMDE5ODdlNjliYjY0NWU0MTQ0ZDg5MDgxN2UwM2U5ZGM2ZWMxN2RiYzVlMGYyN2UxZTc3YTkwYjRhZjM5MTJkIiwKICAicGhhc2UyL2Rldi52Mi5qc29uIjogImU3NDFjMTY1MmY1YjNkY2FlYmM3OGQ5ZGQ5ZDdjMzFmZGY0NDE3MzA2ZmJiMmYwODYyMTQ5ZjI3Zjg1OGYyOGIiLAogICJwaGFzZTIvdHJhaW5fQS52Mi5qc29uIjogImI5NjU2YThjNDFhZjJhNzk1YmNlOWMxMzFlZmM2YTVlZmM2NTJkZjc4MzY4ZDE0ZmI3NzIwZmQ5ODU4ODM2NjYiLAogICJwaGFzZTIvdHJhaW5fQi52Mi5qc29uIjogImM5NzVmZTEyNDU5ZGY2MTYwZWZmYzVhZWIwMDg3Zjk0NjY2YmEwYjQ5Nzc4YzNkODI3YTY3YmI5MjNmZjM1YmQiLAogICJwaGFzZTJfYW5hbHlzaXMvX19pbml0X18ucHkiOiAiYjcyZGY0MTVhMWNjODRlMzA3YjgxZmVlMGIxM2U2ZTNjMWU3YzA0NDQ4YTk1NWNmZmRmNWNjODJjMmI3Nzg4MyIsCiAgInBoYXNlMl9hbmFseXNpcy9hbmFseXplLnB5IjogIjA0MjYzZmVmMDk4ODAyZmQ4ZDNiYjlmMzdmOTNlZWZiYmJlNjU5YWFkMWM3YjUyMzVlMzBkOTMwM2JkZjAwYTQiLAogICJwaGFzZTJfYW5hbHlzaXMvY2FsaWJyYXRlLnB5IjogIjk2YzJmNzRlYzZiNWRlYTM2ZGNhMGE0ZjExZmQwZWI2ZWNmNTlkMzg4NGI1Y2RhZGE3YjViZDczMTExOTFkNTgiLAogICJzZWNvbmRhcnlfYW5hbHlzaXMvX19pbml0X18ucHkiOiAiZjA2NjdmYjMyODQ3MjgwNjMwODU2MGIzYTIzNzEwN2U1ZDVjZGE3ZWU3YWYwMWIzNWZlYWY4ZWRjMGZlM2MyNSIsCiAgInNlY29uZGFyeV9hbmFseXNpcy9jbWkucHkiOiAiNjU0OWE3M2RiMDg3NWI5NTQ2M2UxNWU0Njg1NjJjOGQ5NjM2OWEzZGM2MTJkYjgzYTQwZDQ1OGYwZTc3OTQ4OSIsCiAgInNlY29uZGFyeV9hbmFseXNpcy9udW1lcmljX2VuZHBvaW50LnB5IjogImNlNzBhZWZkMWY0YzAzZmZlODBiYmQwNmQ0ZGYwMjVhYmU5MmZjMDg2NTczMmI2ZDJlNmRiMTM1MDRlZDg4M2YiLAogICJzZWNvbmRhcnlfYW5hbHlzaXMvbnVtZXJpY19saXRlcmFsX3YyLnB5IjogIjgwMjY5YWM2MmQwOWNlMmJkZWUzMjcxZTllZDI1M2FkNGE1MmVjNTFkZTEwNWMzZWQ3ZGM5MTlkODFhNzEyYjQiLAogICJzZWNvbmRhcnlfYW5hbHlzaXMvcmVwb3J0X3NlY29uZGFyeS5weSI6ICJiZmExYWRiYTJiZWZkZDc3NDU5MmI1NzM5NjZiNWMwNjMxOWMzMjU0YzBjZGM1YTljMGVmMjg0YTAxMDdiYTBjIiwKICAidDRfc21va2UvbGFiZWxzLnB5IjogIjIzNmJiM2E3MGY1ZTc0ZGViMmNjMDY3NzA4MjNkOTA5YzBkZjE4ZGIyZTliYjlmM2VhZjA1ZjUxYmRhNjlhMzciLAogICJ0NF9zbW9rZS9waGFzZTJfZXZhbC5weSI6ICIyMmQwMzNiNGNiNjUwYzhkYTA1OTIyMGY2YzdhZWVhOThkNjc2ZGMyYWYzNWRmNWM1MTE5NzE2MjFjNTNiZWJiIiwKICAidDRfc21va2UvcHJvdG9jb2wucHkiOiAiZTY4Mzg0MTZhNWE2YzM5OWYxOWViZGQwYTc2MGM3ZmQwYTEwMTQwZGQ2ZGUwNGFhMzA5MjYwOTNjYzUzN2YxNiIsCiAgInQ0X3Ntb2tlL3RyYWluX3BoYXNlMi5weSI6ICIyYWM4MzIxZjYxYmU0ZGE1NTZkNDI2OTQ4NjIyYTc2ZmYxNWRlMDI0MGQzZTI3ZmU3NzI5MTE1M2I4NjkyMTdiIiwKICAidDRfc21va2UvdHJhaW5fc21va2UucHkiOiAiMDVmYTk0ZmRkZGExMmI5Zjk5YzA1MGNiM2ViMDI1NjI4NzVlYTBmM2ZkMzEyMDQxOTEzNjQzNjdkZDQxNWY0NyIsCiAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L19faW5pdF9fLnB5IjogImY4MzA2MDAyZjViZGExZDI3NTM1ZTFlNmE0YzQyNTVlMjJiMDQ1NDg5ZmI4MDdiMGE3NDYwZTVlMzA4MDM2Y2YiLAogICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9kYXRhLnB5IjogIjUxOWY4OTk1YTYwY2QxOTMzNzVlYWVjZWUyZjg5ODAxYTJlZWE4ZDA2NDc0MjA5ZWFiNDA1NzQ5ODg2YTE2MmMiLAogICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9lbmdpbmUucHkiOiAiODA3ZTYxMDFkMDg0YWY1NWFiODEzNGEyNDE0OTEzZDNjZTAzNjNmZDg5NzJlNmEzMTJlMzVjYzkwODZiYjg4YiIsCiAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L2hmX2JhY2tlbmQucHkiOiAiMWY4N2Q0NTQwNzg1OWViZTI1NGM1MTlmMTkyOTNlMWI1ZmE3NTJiMGY4OWI1NjQzNGU4NDJkYzcwYmIxNmQ4MiIsCiAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L3JlcG9ydC5weSI6ICIwMmRkYzUxNTUwZWFmNThjYmM0NTE1MGQ4ZGUzODMzNzRjNmZiOWM0YWNlNmQ5MTVhMTMzZWVjZWQ5NGU2OGM0IiwKICAidGVzdDFfcGlsb3QvaGluZ2xpc2hfcGlsb3Qvc2NvcmVyX3YxLnB5IjogImM0OGY5ZDEyZTczM2ViMWJmZTVhZjViMWZlY2U2ZTNmZjY5Y2IyMzQyYTk3ZmUzYmE4YWM5ZjAxZjY4ZGJiYmMiCiB9LAogInByZXJlZ2lzdGVyZWRfaGFzaGVzX3ZlcmlmaWVkIjogewogICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9zY29yZXJfdjEucHkiOiAiYzQ4ZjlkMTJlNzMzZWIxYmZlNWFmNWIxZmVjZTZlM2ZmNjljYjIzNDJhOTdmZTNiYThhYzlmMDFmNjhkYmJiYyIsCiAgInNlY29uZGFyeV9hbmFseXNpcy9udW1lcmljX2VuZHBvaW50LnB5IjogImNlNzBhZWZkMWY0YzAzZmZlODBiYmQwNmQ0ZGYwMjVhYmU5MmZjMDg2NTczMmI2ZDJlNmRiMTM1MDRlZDg4M2YiCiB9LAogIm5vdGVib29rX3BheWxvYWRzIjogewogICJQaGFzZTJfQmFzZWxpbmVfREVWLmlweW5iIjogImQzN2FiMmUxZDZhNmJjMmIzZDk2NDFlYTdjNWQwMjc2YjExNjM5ZjEzZjk0OTBiZjhjZDVlMDUzZDZjZDIyNGIiLAogICJQaGFzZTJfVHJhaW4uaXB5bmIiOiAiMDJmYzk1ZTMwMmNhODA3OTRmYjE1NTA1NTAyZWU4YWEwMmIxMDg4YjRiNTNiYzE2MzI3YWVjYTYyNzg0YjFhYyIKIH0sCiAibm90ZWJvb2tfcGF5bG9hZF9ub3RlIjogIlNIQS0yNTYgb2YgdGhlIGNvZGUrZGF0YSBaSVAgZW1iZWRkZWQgaW4gZWFjaCBub3RlYm9vazsgdGhlIGNlbGxzIGFyZSByZW5kZXJlZCBieSBjb2xhYl9ub3RlYm9va3MvYnVpbGRfcGhhc2UyX25vdGVib29rcy5weSAoaGFzaGVkIGFib3ZlKS4gUGhhc2UyX0ZpbmFsVGVzdC5pcHluYiBpcyBidWlsdCBvbmx5IGFmdGVyIEFEQVBURVJTX0ZST1pFTi5tZCwgZnJvbSB0aGUgc2FtZSBidWlsZGVyLCBhbmQgdmVyaWZpZXMgdGhpcyBtYW5pZmVzdC4iLAogImV4cG9zdXJlX29yZGVyIjogewogICJwYXRoIjogInBoYXNlMi9FWFBPU1VSRV9PUkRFUi5qc29uIiwKICAic2hhMjU2IjogImFkZjQzZTUwYzgwYzU5ZmZhODMxMDcxNTU4Nzg1NDY5YjFlZGE4OTg4MGMxOGVlZmI0Zjg5YTlhZGViNWFmYjYiLAogICJvcmRlcl9zaGEyNTYiOiB7CiAgICIxMSI6ICJiZTlhYTkwYjc2MzI3M2ZkMjM5YTkzZGRlZmIwMzVjYTljOWUyMjAxNzQ5OTU2NTJlMjQ2Zjc4ODc3N2MzYmE3IiwKICAgIjIyIjogImY5Y2MwZWI3YmY3ZDI0OTFiY2I0NGI0ZGExMDQ1MmYzODA0OWY0OGIzNWUyMjJiYmVlYTU5MzQ1ZWQ1NTIzNWUiLAogICAiMzMiOiAiOWVjZjJkYzE0OTI2NDg2MzIzM2U4YThkOGY3OWI5OGZiMTE3OGU2ZGQ3MjVlYzJkODQ2ZGRlYmE5YWE5MGEwMyIsCiAgICI0NCI6ICIzYzZhNjY0N2FkZjU2YTllYmEzODNiMmRjM2VjN2RiN2VjMGVkOGFkZmE0ZDc2OWI2M2VmYzM3N2FkZDBhYzdjIiwKICAgIjU1IjogImQ0YmM4YzgwOWRkYzgwNTY1Njc5ZWVhMGY5MzUzNjEyZWNjNmFhMjNiZDNkYzNjMzg3MTE3YzA1ODRlZDAwZDkiCiAgfQogfSwKICJwcm9tcHQiOiB7CiAgInZlcnNpb24iOiAidGVzdDEtZW5nbGlzaC1maW5hbC12MiIsCiAgImluc3RydWN0aW9uX3NoYTI1NiI6ICIzZDBiNTE2YzlmYjU5Nzg3NTVlMTJiZGQyYThiM2MxNzQyYjYwNWUyYzNlZGMyZTM5ZmMyYzUzMTE1ZGM3ZDYzIiwKICAiaWRlbnRpY2FsX2luIjogWwogICAidDRfc21va2UvcHJvdG9jb2wucHkiLAogICAidGVzdDFfcGlsb3QvaGluZ2xpc2hfcGlsb3QvZW5naW5lLnB5IgogIF0KIH0sCiAibW9kZWwiOiB7CiAgImlkIjogIlF3ZW4vUXdlbjMtNEIiLAogICJyZXZpc2lvbiI6ICIxY2ZhOWE3MjA4OTEyMTI2NDU5MjE0ZThiMDQzMjE2MDNiM2RmNjBjIiwKICAidG9rZW5pemVyX3JldmlzaW9uIjogIjFjZmE5YTcyMDg5MTIxMjY0NTkyMTRlOGIwNDMyMTYwM2IzZGY2MGMiLAogICJodWJfZmlsZXMiOiB7CiAgICJjb25maWcuanNvbiI6ICI4YmEwMDZmNzRmZWNmYWFlYjM5Mjg3MmE2MGY0YTQ4MGU3ZWM5ODYwMTUzZDJlMWI3NjllYzgxZjlhMTQ3ZjhhIiwKICAgImdlbmVyYXRpb25fY29uZmlnLmpzb24iOiAiMjMyNWRhMGYxNWJiODQ4ZTAxOGM1YWUwNzFiNzk0MzMzMmU5Zjg3MWQ2YjYwZTJlZDIyY2E5N2Q0Y2I5OTNkMiIsCiAgICJtZXJnZXMudHh0IjogIjg4MzFlNGYxYTA0NDQ3MTM0MGY3YzBhODNkN2JkNzEzMDZhNWI4NjdlOTVmZDg3MGY3NGQwYzUzMDhhOTA0ZDUiLAogICAibW9kZWwuc2FmZXRlbnNvcnMuaW5kZXguanNvbiI6ICI2ZGMwOTgxYjg4MjlmZWFkNzQ2NDQxZjY4ZjM4ZjI0YzVjYTRhM2E2NjM1MWY2NTJjMjZjNmRmMGVmYzQzYWIyIiwKICAgInRva2VuaXplci5qc29uIjogImFlYjEzMzA3YTcxYWNkOGZlODE4NjFkOTRhZDU0YWI2ODlkZjc3MzMxODgwOWVlZDNjYmU3OTRiNDQ5MmRhZTQiLAogICAidG9rZW5pemVyX2NvbmZpZy5qc29uIjogImQ1ZDA5ZjA3YjQ4YzMwODZjNTA4YjMwZDFjOTExNGJkMTE4OTE0NWI3NGU5ODJhMjY1MzUwYzkyM2FjZDgxMDEiLAogICAidm9jYWIuanNvbiI6ICJjYTEwZDdlOWZiM2VkMTg1NzVkZDFlMjc3YTI1NzljMTZkMTA4ZTMyZjI3NDM5Njg0YWZhMGUxMGIxNDQwOTEwIgogIH0sCiAgImNoYXRfdGVtcGxhdGUiOiB7CiAgICJzb3VyY2UiOiAidG9rZW5pemVyX2NvbmZpZy5qc29uIGF0IHRoZSBwaW5uZWQgcmV2aXNpb24gKG5vIGNoYXRfdGVtcGxhdGUuamluamEgZXhpc3RzIHRoZXJlKSIsCiAgICJkaWdlc3QiOiAiNDFkNTkyOWJmNzM3OTZiZWI2NjgwOWFjNzAwYjJjZjNmZjgxNjk0ZjkzM2U1YzE0ZDUyYjdmZDY5NjNjOTQ3ZCIsCiAgICJkaWdlc3RfZGVmaW5pdGlvbiI6ICJ0cmFpbl9zbW9rZS5kaWdlc3QodGVtcGxhdGUpID0gU0hBLTI1NiBvZiBqc29uLmR1bXBzKHRlbXBsYXRlKSBpbiBVVEYtODsgcmVjb3JkZWQgYXQgcnVuIHRpbWUgYXMgY2hhdF90ZW1wbGF0ZV9zaGEyNTYgYnkgdGhlIHRyYWluZXIgYW5kIGJ5IHRoZSBlbmdpbmUgYmFja2VuZCIsCiAgICJ1dGY4X3NoYTI1NiI6ICJhNTVlZTFiMTY2MDEyOGI3MDk4NzIzZTBhYmNkOTJjYWEwNzg4MDYxMDUxYzYyZDUxY2JlODdkOWNmMTk3NGQ4IiwKICAgImt3YXJncyI6IHsKICAgICJlbmFibGVfdGhpbmtpbmciOiBmYWxzZSwKICAgICJhZGRfZ2VuZXJhdGlvbl9wcm9tcHQiOiB0cnVlCiAgIH0KICB9LAogICJ0cnVzdF9yZW1vdGVfY29kZSI6IGZhbHNlLAogICJzZWNvbmRfYmFja2JvbmUiOiAibm9uZSAoQW1lbmRtZW50IDEgaXRlbSAxNTogTGxhbWEtMy4yLTNCIGV4Y2x1ZGVkKSIKIH0sCiAiZ2VuZXJhdGlvbl9jb25maWciOiB7CiAgImVuZ2luZV9jb25maWdfc2hhMjU2IjogewogICAiZGV2IjogImU4MjgwNjIyZjljZjQ1ZmJiNjJiYjUzN2MxODgzYzViMzk0YTdhOGQyM2U4NTE2NGRiYjdiMjk3ZWVjOGY1YjYiLAogICAiZmluYWxfdGVzdCI6ICJkNzI5YWRhNDk3Y2RkNjE0ZjI1ODY1OGEzOTczYTJiNzkzMDVmNGUxODIxOTIyMDIwOGJlNTdkZDM0OTJiY2E4IgogIH0sCiAgImRlY29kaW5nIjogewogICAibW9kZWxfaWQiOiAiUXdlbi9Rd2VuMy00QiIsCiAgICJzZWVkIjogMTEsCiAgICJtYXhfbmV3X3Rva2VucyI6IDEwMjQsCiAgICJjb250ZXh0X2xpbWl0IjogMjA0OCwKICAgInRoaW5raW5nIjogZmFsc2UsCiAgICJkb19zYW1wbGUiOiBmYWxzZSwKICAgIm51bV9iZWFtcyI6IDEsCiAgICJxdWFudGl6YXRpb24iOiAibmY0IiwKICAgImRvdWJsZV9xdWFudGl6YXRpb24iOiB0cnVlLAogICAiY29tcHV0ZV9kdHlwZSI6ICJmbG9hdDE2IiwKICAgImF0dGVudGlvbl9iYWNrZW5kIjogInNkcGFfbWF0aCIsCiAgICJtYXhfaW5mcmFzdHJ1Y3R1cmVfcmV0cmllcyI6IDEsCiAgICJwcm9tcHRfdmVyc2lvbiI6ICJ0ZXN0MS1lbmdsaXNoLWZpbmFsLXYyIgogIH0sCiAgImhmX2dlbmVyYXRpb25fY29uZmlnIjogewogICAibWF4X25ld190b2tlbnMiOiAxMDI0LAogICAiZG9fc2FtcGxlIjogZmFsc2UsCiAgICJudW1fYmVhbXMiOiAxLAogICAidXNlX2NhY2hlIjogdHJ1ZSwKICAgImVvc190b2tlbl9pZCI6ICJtb2RlbC5nZW5lcmF0aW9uX2NvbmZpZy5lb3NfdG9rZW5faWQiLAogICAicGFkX3Rva2VuX2lkIjogMTUxNjQzLAogICAicmVwZXRpdGlvbl9wZW5hbHR5IjogMS4wLAogICAidGVtcGVyYXR1cmUiOiAxLjAsCiAgICJ0b3BfcCI6IDEuMCwKICAgInRvcF9rIjogNTAKICB9LAogICJlbmdpbmVfc2VlZCI6IDExLAogICJyZXRyeV9wb2xpY3kiOiAiYXQgbW9zdCBvbmUgdW5jaGFuZ2VkLWNvbmZpZ3VyYXRpb24gaW5mcmFzdHJ1Y3R1cmUgcmV0cnkgcGVyIHByb21wdDsgYSBzdWNjZXNzZnVsbHkgZ2VuZXJhdGVkIHJlc3BvbnNlIGlzIG5ldmVyIHJldHJpZWQ7IE9PTSBuZXZlciByZXRyaWVkIiwKICAiZmluYWxfdGVzdF9vdXRwdXQiOiAic2VhbGVkOiBjb21wbGV0aW9uLCBjaGVja3N1bSBhbmQgaW5mcmFzdHJ1Y3R1cmUgc3RhdHVzIG9ubHkgKEFtZW5kbWVudCAxIGl0ZW0gMTQpIgogfSwKICJlb3Nfc3RvcHBpbmciOiB7CiAgInRyYWluaW5nX3RhcmdldCI6ICJyZWZlcmVuY2Vfc29sdXRpb25fZW4gKyAnPHxpbV9lbmR8PicgKGlkIDE1MTY0NSksIGVuY29kZWQgc2VwYXJhdGVseSBmcm9tIHRoZSBwcm9tcHQ7IHRoZSBsYXN0IHN1cGVydmlzZWQgdG9rZW4gaXMgYXNzZXJ0ZWQgdG8gYmUgPHxpbV9lbmR8PiIsCiAgImdlbmVyYXRpb25fZW9zX3Rva2VuX2lkIjogWwogICAxNTE2NDUsCiAgIDE1MTY0MwogIF0sCiAgImdlbmVyYXRpb25fZW9zX3NvdXJjZSI6ICJtb2RlbC5nZW5lcmF0aW9uX2NvbmZpZy5lb3NfdG9rZW5faWQgZnJvbSBnZW5lcmF0aW9uX2NvbmZpZy5qc29uIGF0IHRoZSBwaW5uZWQgcmV2aXNpb24gKDx8aW1fZW5kfD4gMTUxNjQ1LCA8fGVuZG9mdGV4dHw+IDE1MTY0Myk7IHBhZF90b2tlbl9pZCA9IHRva2VuaXplciBwYWQgPHxlbmRvZnRleHR8PiAxNTE2NDMiLAogICJzdG9wcGluZyI6ICJncmVlZHkgZGVjb2Rpbmcgc3RvcHMgYXQgdGhlIGZpcnN0IGdlbmVyYXRlZCBFT1MgaWQgb3IgYWZ0ZXIgbWF4X25ld190b2tlbnMgPSAxMDI0IG5ldyB0b2tlbnMiLAogICJ0cnVuY2F0ZWQiOiAibm8gRU9TIGFtb25nIHRoZSBnZW5lcmF0ZWQgaWRzIGFuZCAxMDI0IHRva2VucyBnZW5lcmF0ZWQ7IHNjb3JlZCAwIGJ5IHNjb3JlciB2MSIsCiAgImRlY29kaW5nX29mX3Jlc3BvbnNlIjogInRva2VuaXplci5kZWNvZGUoaWRzLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpOyB0aGUgcmF3IGZvcm0gd2l0aCBzcGVjaWFsIHRva2VucyBpcyBhbHNvIHN0b3JlZCIsCiAgInByb21wdCI6ICJjaGF0IHRlbXBsYXRlIHdpdGggc3lzdGVtID0gZnJvemVuIGluc3RydWN0aW9uLCB1c2VyID0gcmVuZGVyaW5nLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSwgZW5hYmxlX3RoaW5raW5nPUZhbHNlIChlbXB0eSA8dGhpbms+PC90aGluaz4gYmxvY2spOyBpZGVudGljYWwgYXQgdHJhaW5pbmcgYW5kIGV2YWx1YXRpb24iCiB9LAogImR0eXBlX3ByZXBhcmF0aW9uIjogewogICJ0cmFpbiI6IFsKICAgIkF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZCh0b3JjaF9kdHlwZT1mbG9hdDE2LCBCaXRzQW5kQnl0ZXNDb25maWcgbmY0LCBkb3VibGUgcXVhbnQsIGNvbXB1dGUgZmxvYXQxNiksIHNkcGEgYXR0ZW50aW9uLCBkZXZpY2VfbWFwIHsnJzogMH0iLAogICAicGVmdC5wcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nKHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUsIHVzZV9yZWVudHJhbnQ9RmFsc2UpOiBjYXN0cyB0aGUgbm9uLXF1YW50aXplZCBwYXJhbWV0ZXJzIChub3JtcywgdGllZCBlbWJlZGRpbmcvbG1faGVhZCkgdG8gZmxvYXQzMiIsCiAgICJMb1JBIHIxNi9hbHBoYTMyL2Ryb3BvdXQgMC4wNSBvbiBxL2svdi9vL2dhdGUvdXAvZG93bjsgdHJhaW5hYmxlIHBhcmFtZXRlcnMgZm9yY2VkIHRvIGZsb2F0MzIiLAogICAiZm9yd2FyZCB1bmRlciB0b3JjaC5hdXRvY2FzdCgnY3VkYScsIGZsb2F0MTYpOyBsb3NzIGluIGZsb2F0MzI7IEFkYW1XIHN0YXRlIGZsb2F0MzI7IEdyYWRTY2FsZXIgaW5pdCA2NTUzNiwgZ3Jvd3RoIDIuMCwgYmFja29mZiAwLjUsIGludGVydmFsIDIwMDA7IFNEUEEgbWF0aCBrZXJuZWwgb25seTsgdXNlX2NhY2hlIEZhbHNlIgogIF0sCiAgImV2YWx1YXRpb24iOiBbCiAgICJBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQodG9yY2hfZHR5cGU9ZmxvYXQxNiwgQml0c0FuZEJ5dGVzQ29uZmlnIG5mNCwgZG91YmxlIHF1YW50LCBjb21wdXRlIGZsb2F0MTYpLCBzZHBhIGF0dGVudGlvbiwgbW9kZWwuZXZhbCgpLCBubyBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nIChub3JtcyBhbmQgZW1iZWRkaW5ncyBzdGF5IGZsb2F0MTYpIiwKICAgIlBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoYmFzZSwgYWRhcHRlciwgaXNfdHJhaW5hYmxlPUZhbHNlKTogdGhlIGZsb2F0MzIgTG9SQSB3ZWlnaHRzIGFyZSBsb2FkZWQgdW5tZXJnZWQgKG5vIG1lcmdlLCBubyByZS1xdWFudGl6YXRpb24pOyBubyBhdXRvY2FzdCIsCiAgICJpbmZlcmVuY2VfbW9kZSwgU0RQQSBtYXRoIGtlcm5lbCBvbmx5LCB1c2VfY2FjaGUgVHJ1ZSwgYmF0Y2ggMSwgZGV0ZXJtaW5pc3RpYyBhbGdvcml0aG1zIgogIF0sCiAgImRpZmZlcmVuY2UiOiAidHJhaW5pbmcgdXBjYXN0cyBub3Jtcy9lbWJlZGRpbmdzIHRvIGZsb2F0MzIgYW5kIHJ1bnMgdW5kZXIgZmxvYXQxNiBhdXRvY2FzdDsgZXZhbHVhdGlvbiBrZWVwcyB0aGUgZnJvemVuIGVuZ2luZSdzIGZsb2F0MTYgYmFzZS4gU3RhbmRhcmQgUUxvUkEgcHJhY3RpY2UsIGZpeGVkIGJlZm9yZSBhbnkgUGhhc2UtMiBvdXRwdXQuIgogfSwKICJ0cmFpbmluZyI6IHsKICAicmVjaXBlX3NoYTI1NiI6ICJiNzM4NDlmZTY0OTI2MmE3YWZlZTA1ZWViYjk3ZWY3YTYzNTUxNDFkY2Q1NGYzODViMDEwMTk1ZDI5Njg3OWI0IiwKICAiZ3JhZF9zY2FsZXIiOiB7CiAgICJpbml0X3NjYWxlIjogNjU1MzYuMCwKICAgImdyb3d0aF9mYWN0b3IiOiAyLjAsCiAgICJiYWNrb2ZmX2ZhY3RvciI6IDAuNSwKICAgImdyb3d0aF9pbnRlcnZhbCI6IDIwMDAKICB9LAogICJ2YWxpZF9ydW4iOiAiZXhhY3RseSA2MCBhY2N1bXVsYXRpb24gd2luZG93cyBhbmQgNjAgc3VjY2Vzc2Z1bCBvcHRpbWl6ZXIgc3RlcHMgb3ZlciBleGFjdGx5IDMgcGFzc2VzOyBscih1KSBpcyBzZXQgYmVmb3JlIHN0ZXAgdTsgZ3JhZGllbnRzIGFyZSB1bnNjYWxlZCBiZWZvcmUgY2xpcHBpbmc7IGFueSBHcmFkU2NhbGVyIHNraXAsIG5vbi1maW5pdGUgbG9zcyBvciBncmFkaWVudCwgb3Igbm9uLWZpbml0ZSBzYXZlZCBwYXJhbWV0ZXIgaXMgYSBudW1lcmljYWwgZmFpbHVyZSAoQW1lbmRtZW50IDEgaXRlbSAxMCkiLAogICJmYWlsdXJlX3BvbGljeSI6ICJBbWVuZG1lbnQgMSBpdGVtIDExOiBpbmZyYXN0cnVjdHVyZSAtPiBvbmUgZnJvbS1zY3JhdGNoIHJldHJ5OyBudW1lcmljYWwvcHJvdG9jb2wgLT4gc3R1ZHkgaGFsdHMiCiB9LAogImRlcGVuZGVuY2llcyI6IHsKICAicHl0aG9uIjogInJlY29yZGVkIHBlciBydW4gKENvbGFiIGRlZmF1bHQpIiwKICAidG9yY2giOiAiMi4xMS4wK2N1MTMwIChDb2xhYiBidWlsZDsgcmVjb3JkZWQsIG5vdCBpbnN0YWxsZWQpIiwKICAidHJhaW4iOiB7CiAgICJ0cmFuc2Zvcm1lcnMiOiAiNC41MS4zIiwKICAgImJpdHNhbmRieXRlcyI6ICIwLjUwLjIiLAogICAicGVmdCI6ICIwLjE1LjIiLAogICAiYWNjZWxlcmF0ZSI6ICIxLjYuMCIsCiAgICJodWdnaW5nZmFjZS1odWIiOiAiMC4zMC4yIiwKICAgInRva2VuaXplcnMiOiAiMC4yMS4xIiwKICAgInNhZmV0ZW5zb3JzIjogIjAuNS4zIiwKICAgIm52aWRpYS1tbC1weSI6ICIxMi41NzAuODYiCiAgfSwKICAiZXZhbHVhdGlvbiI6IHsKICAgInRyYW5zZm9ybWVycyI6ICI0LjUxLjMiLAogICAiYml0c2FuZGJ5dGVzIjogIjAuNTAuMiIsCiAgICJhY2NlbGVyYXRlIjogIjEuNi4wIiwKICAgImh1Z2dpbmdmYWNlLWh1YiI6ICIwLjMwLjIiLAogICAidG9rZW5pemVycyI6ICIwLjIxLjEiLAogICAic2FmZXRlbnNvcnMiOiAiMC41LjMiCiAgfSwKICAiYWRhcHRlcl9ldmFsdWF0aW9uIjogewogICAicGVmdCI6ICIwLjE1LjIiCiAgfQogfQp9CmBgYAo=').decode('utf-8')
MANIFEST_SHA256 = '6a6370cf9bb8dec6d31a020031bb3ec389c5d5ae4b89fcece465b2464918ab19'
assert hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest() == MANIFEST_SHA256, 'Embedded IMPLEMENTATION_MANIFEST.md was changed: refusing to run'
MANIFEST = json.loads(MANIFEST_TEXT.split('```json\n', 1)[1].split('\n```', 1)[0])
assert MANIFEST['preregistration']['sha256'] == '37ef68bfcb7cfa0bb530bfbecbf2082d18e916c0217370085d4180e8ea759ada', 'Manifest is for another PREREGISTRATION.md: refusing to run'
assert MANIFEST['notebook_payloads']['Phase2_Baseline_DEV.ipynb'] == hashlib.sha256(blob).hexdigest(), 'Notebook payload differs from the implementation manifest: refusing to run'
for member, source in {'hinglish_pilot/__init__.py': 'test1_pilot/hinglish_pilot/__init__.py', 'hinglish_pilot/data.py': 'test1_pilot/hinglish_pilot/data.py', 'hinglish_pilot/engine.py': 'test1_pilot/hinglish_pilot/engine.py', 'hinglish_pilot/hf_backend.py': 'test1_pilot/hinglish_pilot/hf_backend.py', 'hinglish_pilot/report.py': 'test1_pilot/hinglish_pilot/report.py', 'hinglish_pilot/scorer_v1.py': 'test1_pilot/hinglish_pilot/scorer_v1.py', 'secondary_analysis/__init__.py': 'secondary_analysis/__init__.py', 'secondary_analysis/cmi.py': 'secondary_analysis/cmi.py', 'secondary_analysis/numeric_endpoint.py': 'secondary_analysis/numeric_endpoint.py', 'secondary_analysis/report_secondary.py': 'secondary_analysis/report_secondary.py', 't4_smoke/train_smoke.py': 't4_smoke/train_smoke.py', 't4_smoke/labels.py': 't4_smoke/labels.py', 't4_smoke/protocol.py': 't4_smoke/protocol.py', 't4_smoke/train_phase2.py': 't4_smoke/train_phase2.py', 't4_smoke/phase2_eval.py': 't4_smoke/phase2_eval.py', 'inputs/dev.v2.json': 'phase2/dev.v2.json'}.items():
    assert hashlib.sha256((PACKAGE / member).read_bytes()).hexdigest() == MANIFEST['files'][source], f'{member} differs from the implementation manifest: refusing to run'
(INPUTS / 'IMPLEMENTATION_MANIFEST.md').write_text(MANIFEST_TEXT, encoding='utf-8')
print('Implementation manifest verified:', MANIFEST_SHA256)
if USE_DRIVE and Path('/content').exists():  # Drive only inside Colab
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = Path('/content/drive/MyDrive/Hinglish_Phase2')
else:
    STORE = PACKAGE / 'runs'
STORE.mkdir(parents=True, exist_ok=True)
try:  # Qwen3-4B is ungated; a token only avoids anonymous rate limits.
    from google.colab import userdata
    os.environ['HF_TOKEN'] = (userdata.get('HF_TOKEN') or '').strip() or os.environ.get('HF_TOKEN', '')
except Exception as error:
    print('No HF_TOKEN Colab secret (' + type(error).__name__ + '); downloading anonymously.')
if not os.environ.get('HF_TOKEN'):
    os.environ.pop('HF_TOKEN', None)

def step(log_dir, script, *args):
    """Run one phase in a fresh Python process, stream its output and log the attempt."""
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', str(PACKAGE / 't4_smoke' / script), *map(str, args)]
    started, returncode = time.time(), None
    try:
        with open(log_dir / 'log.txt', 'a') as log, subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                                     text=True, errors='replace') as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
            except BaseException:  # interrupt: never leave a phase running on the GPU behind the notebook
                process.kill()
                process.wait()
                log.write('[notebook] interrupted; phase process killed\n')
                raise
        returncode = process.returncode
    finally:
        with open(log_dir / 'attempts.jsonl', 'a') as attempts:
            attempts.write(json.dumps({'script': script, 'args': list(map(str, args)), 'returncode': returncode,
                                       'started': started, 'seconds': time.time() - started}) + '\n')
    return returncode

def zip_results(folder, archive, prefix):
    """All small files under ``folder``; weights, optimizer state and per-attempt journals stay out (hashes listed)."""
    files, large = [], {}
    for path in sorted(Path(folder).rglob('*')):
        if not path.is_file() or 'journal' in path.relative_to(folder).parts or path.name.endswith('.lock'):
            continue
        if path.suffix in ('.pt', '.safetensors'):
            large[str(path.relative_to(folder))] = hashlib.sha256(path.read_bytes()).hexdigest()
        else:
            files.append(path)
    sums = ''.join(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(folder)}\n' for p in files)
    sums += ''.join(f'{digest}  {name}  (not included)\n' for name, digest in large.items())
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in files:
            z.write(path, str(Path(prefix) / path.relative_to(folder)))
        z.writestr(str(Path(prefix) / 'SHA256SUMS.txt'), sums)
    print(sums)
    print('Return:', archive, 'SHA-256', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
    try:
        from google.colab import files as colab_files
        colab_files.download(str(archive))
    except ImportError:
        print('Local: ZIP saved on disk.')


In [ ]:
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:  # CPU runtime: no nvidia-smi at all
    gpu = ''
print('GPU:', gpu or 'none')
assert 'T4' in gpu, 'Select Runtime → Change runtime type → T4 GPU'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements-phase2.txt')], check=True)
# A fresh interpreter sees the installed versions; this kernel never imports torch.
STACK = json.loads(subprocess.run([sys.executable, '-c', 'import sys; sys.path.insert(0, sys.argv[1]); import train_smoke as t, json; '
                                   'v = t.versions(); print(json.dumps({"versions": v, "mismatches": t.stack_mismatches(v)}))',
                                   str(PACKAGE / 't4_smoke')], capture_output=True, text=True, check=True).stdout)
print(json.dumps(STACK['versions'], indent=1))
STACK_MISMATCHES = STACK['mismatches']
print('D021 stack mismatches:', STACK_MISMATCHES or 'none')
# Tokenizer, chat template and config files at the pinned revision must match the implementation manifest.
HUB_CHECK = """
import hashlib, json, os, sys
from huggingface_hub import hf_hub_download
model = json.loads(sys.argv[1])
for name, expected in sorted(model['hub_files'].items()):
    path = hf_hub_download(model['id'], name, revision=model['revision'], token=os.environ.get('HF_TOKEN') or None)
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert actual == expected, f'{name} at the pinned revision has SHA-256 {actual}, manifest {expected}: refusing to run'
print('Pinned Hub files match the implementation manifest:', ', '.join(sorted(model['hub_files'])))
"""
print(subprocess.run([sys.executable, '-c', HUB_CHECK, json.dumps(MANIFEST['model'])], capture_output=True, text=True, check=True).stdout)


In [ ]:
RUN_DIR = STORE / 'baseline_dev' / 'base'
step(RUN_DIR.parent, 'phase2_eval.py', 'evaluate', '--split', 'dev', '--inputs', INPUTS, '--run-dir', RUN_DIR)
FLAG = RUN_DIR.parent / 'base_ceiling_flag.json'
assert FLAG.exists(), 'Evaluation incomplete (see above): re-run all to resume, then download'
flag = json.loads(FLAG.read_text())
low, high = flag['wilson95']
print(f"\n{flag['name']}: {'FIRES' if flag['flag'] else 'does not fire'} — unchanged DEV C {flag['k']}/{flag['n']}"
      f" (Wilson 95% {100 * low:.1f}–{100 * high:.1f}%; threshold >= 38/40)")


## Download
If the run stopped early, re-run all cells first. Then run this cell.


In [ ]:
zip_results(STORE / 'baseline_dev', PACKAGE / 'Phase2_Baseline_DEV_results.zip', 'baseline_dev')
